In [0]:
%sql
-- select everything
select * 
from default.loans

In [0]:
select *
from default.loans 
where loan_id is null 
      or customer_id is null 
      or institution_id is null 
      or loan_amount is null 
      or loan_status is null 
      or interest_rate is null 
      or loan_tenure_months is null 
      or application_date is null 
      or disbursement_date is null
      or maturity_date is null 
      or emi_amount is null 
      or purpose_of_loan is null

In [0]:
select *
from default.loans 
where loan_amount <=0 
      or emi_amount <= 0
      or interest_rate <= 0

In [0]:
select *
from default.loans 
where application_date >= disbursement_date 
      or application_date >= maturity_date 
      or disbursement_date >= maturity_date

In [0]:
select *  
from default.loans 
where loan_amount <= 0
      or interest_rate not between 5 and 20 
      or loan_tenure_months < 0
      

In [0]:
with xyz as 
(select loan_id, round(emi_amount) as emi_amount, 

round(loan_amount * (interest_rate/1200) * power((1 + interest_rate/1200) , loan_tenure_months) / 
(power((1 + interest_rate/1200), loan_tenure_months) - 1)) as emi_amount_calc

from default.loans)

select loan_id, emi_amount, emi_amount_calc
from xyz 
where abs(emi_amount - emi_amount_calc) > 2

In [0]:
-- create silver table

create or replace table default.silver_loans 
as

with silver_cte as
(select 
loan_id,
customer_id,
institution_id,
loan_amount,
loan_status,
interest_rate,
loan_tenure_months,
application_date,
disbursement_date,
maturity_date,
emi_amount,
purpose_of_loan,
round(loan_amount * (interest_rate/1200) * power((1 + interest_rate/1200) , loan_tenure_months) / 
(power((1 + interest_rate/1200), loan_tenure_months) - 1)) as emi_amount_calc

from default.loans)

select 
loan_id,
customer_id,
institution_id,
loan_amount,
loan_status,
interest_rate,
loan_tenure_months,
application_date,
disbursement_date,
maturity_date,
round(emi_amount) as emi_amount,
round(loan_amount * (interest_rate/1200) * power((1 + interest_rate/1200) , loan_tenure_months) / 
(power((1 + interest_rate/1200), loan_tenure_months) - 1)) as emi_amount_calc,
purpose_of_loan

from silver_cte
where loan_id is not null 
      and customer_id is not null 
      and institution_id is not null 
      and loan_amount is not null 
      and loan_status is not null 
      and interest_rate is not null 
      and loan_tenure_months is not null 
      and application_date is not null 
      and disbursement_date is not null
      and maturity_date is not null 
      and emi_amount is not null 
      and purpose_of_loan is not null
      and loan_amount > 0 
      and emi_amount > 0
      and interest_rate > 0
      and application_date < disbursement_date 
      and application_date < maturity_date 
      and disbursement_date < maturity_date
      and loan_amount > 0
      and interest_rate between 5 and 20 
      and loan_tenure_months > 0
      and abs(round(emi_amount) - round(emi_amount_calc)) <= 2

In [0]:
select * 
from default.silver_loans

In [0]:
-- What is the portfolio breakdown by loan status (Active, Defaulted, Overdue, Closed)?

create or replace table loan_distribution_gold 
as 

with loan_distribution as 
(select loan_status, 
       round((sum(loan_amount) / 10000000),2) as portfolio_value_in_Cr,
       count(*) as No_of_loans
from default.silver_loans
group by loan_status
order by sum(loan_amount) desc)

select loan_status, portfolio_value_in_Cr, No_of_loans, 
       round((No_Of_loans * 100.0 / sum(No_of_loans) over()),2) as pct_contribution
from loan_distribution

In [0]:
select * 
from loan_distribution_gold

In [0]:
-- Which customers have multiple loans and what's their repayment behavior?

create or replace table customers_with_multiple_loans_gold
as

with loan_distribution as 
(select customer_id , 
        loan_status, 
        count(*) as No_of_loans_in_that_status, 
        round((sum(loan_amount) / 100000),2) as Total_amount_in_that_status_in_lakhs
from default.silver_loans
group by customer_id , loan_status)

, total_loan_addon as
(select *, sum(No_of_loans_in_that_status) over(partition by customer_id) as Total_loans 
from loan_distribution)

select customer_id,
       loan_status,
       No_of_loans_in_that_status,
       Total_amount_in_that_status_in_lakhs,
       Total_loans
from total_loan_addon
where Total_loans > 1


In [0]:
select * 
from customers_with_multiple_loans_gold

In [0]:
-- What is the portfolio performance by loan purpose (Course Fees, Living Expenses, etc.)?
create or replace table loan_purpose_gold
as 

with loan_purpose as 
(select purpose_of_loan, round((sum(loan_amount) /10000000),2) as Loan_value_in_Cr_per_purpose
from default.silver_loans 
group by purpose_of_loan)

select *, round((Loan_value_in_Cr_per_purpose * 100.0 / sum(Loan_value_in_Cr_per_purpose) over()),2) as pct_contribution
from loan_purpose

In [0]:
select * 
from loan_purpose_gold

In [0]:
-- What are the executive KPIs (total portfolio value, default rate, average interest rate)?
create or replace table executive_KPIs 
as

select round((sum(loan_amount) / 10000000),2) as portfolio_value_in_Cr, 
       round(sum(case when loan_status = 'Defaulted' then loan_amount / 10000000 end),2) as total_default_amount_in_Cr,
       round(sum(case when loan_status = 'Defaulted' then 1 else 0 end),2) as default_loans_count,
       count(distinct loan_id) as total_loans,
       round(sum(case when loan_status = 'Defaulted' then 1 else 0 end) * 100.0 / count(distinct loan_id),2) as default_rate,
       round(avg(interest_rate),2) as avg_interest_rate
from default.silver_loans



In [0]:
select * 
from executive_KPIs

In [0]:
-- How does loan performance vary by loan size (small, medium, large loans)?

create or replace table loan_performance_by_size_gold 
as 

with loan_buckets as 
(select loan_id, round((loan_amount/100000),2) as loan_amount_in_lakhs, loan_status,
      case when loan_amount > 600000 then 'large'
           when loan_amount > 300000 then 'medium'
           else 'small'
      end as loan_size
from default.silver_loans)

select loan_size, loan_status, count(*) as No_of_loans, round(sum(loan_amount_in_lakhs)/100,2) as Total_loan_amount_in_Cr
from loan_buckets
group by loan_size, loan_status
order by loan_size, loan_status

In [0]:
select * 
from loan_performance_by_size_gold

In [0]:
-- Watermark Table Structure:
-- This table tracks the last processed date for each data layer
-- It enables incremental processing by storing a checkpoint

CREATE TABLE IF NOT EXISTS default.watermark (
  layer STRING COMMENT 'Data layer name (bronze, silver, gold)',
  last_processed_date DATE COMMENT 'Last date processed for this layer'
) COMMENT 'Tracks incremental processing checkpoint for each layer';

-- Initialize watermark for silver layer
-- Starting at 2021-01-01 ensures first incremental run processes all existing loans
INSERT INTO default.watermark
VALUES ('silver', '2021-01-01');